# 04 — QAOA depth sweep

**Sweep 1.** Fixed problem — the canonical 16-equity universe with `K=4`.
Vary `p in {1..5}`. For each `p` we run 10 multi-start COBYLA seeds and
keep the best, then report approximation ratio, P(optimum), and
P(feasible) across `p`.

Results cache to `results/depth_sweep.json` so re-plotting is instant.

In [ ]:
# === Colab Setup ===
import os, sys, json
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_url = 'https://github.com/egil10/fys5419.git'
    repo_dir = '/content/fys5419'
    if not os.path.exists(repo_dir):
        get_ipython().system(f'git clone {repo_url} {repo_dir}')
    else:
        get_ipython().system(f'git -C {repo_dir} pull')
    os.chdir(f'{repo_dir}/project2/code/notebooks')

sys.path.append('..')
from scripts.colab import setup, out_dir; setup()

# === Project imports ===
import numpy as np
import pandas as pd

from scripts.data      import load_universe
from scripts.portfolio import PortfolioProblem, DEFAULTS
from scripts.classical import brute_force
from scripts.qaoa      import solve
from scripts.metrics   import prob_optimal, prob_feasible

# === Paths (Drive on Colab, local repo otherwise) ===
RESULTS = out_dir('results')
print(f'Results will be saved to: {RESULTS}')

In [2]:
# === Canonical instance: n=16, K=4 ===
P_VALUES   = [1, 2, 3, 4, 5]
N_RESTARTS = 10

r  = load_universe()
pf = PortfolioProblem(r.mu, r.Sigma,
                      lam=DEFAULTS['lam'], A=DEFAULTS['A'],
                      K=DEFAULTS['K_AT_16'], tickers=r.tickers)
bf = brute_force(pf)
print(f'brute force: {bf.bitstring}  C={bf.cost:.6f}  picks={bf.tickers(pf)}')

brute force: 0010000010010001  C=-0.001404  picks=['GOOGL', 'IBM', 'NVDA', 'JPM']


In [ ]:
cache = RESULTS / 'depth_sweep.json'

if cache.exists():
    sweep = json.loads(cache.read_text())
    print(f'loaded {cache.name}')
else:
    sweep = []
    for p in P_VALUES:
        res = solve(pf, p=p, n_restarts=N_RESTARTS, seed=42)
        sweep.append({
            'p':           p,
            'energy':      float(res['energy']),
            'ratio':       float(res['ratio']),
            'p_optimal':   prob_optimal(res['probs'], bf.x),
            'p_feasible':  prob_feasible(res['probs'], pf.n, pf.K),
            'best_C':      float(min(h['fun'] for h in res['history'])),
        })
        print(f'  p={p}: ratio={res["ratio"]:.4f}  '
              f'P(opt)={sweep[-1]["p_optimal"]:.4f}  '
              f'P(feas)={sweep[-1]["p_feasible"]:.4f}')
    cache.write_text(json.dumps(sweep, indent=2))
    print(f'saved -> {cache.name}')

pd.DataFrame(sweep)

  p=1: ratio=0.9910  P(opt)=0.0002  P(feas)=0.3812
  p=2: ratio=0.9927  P(opt)=0.0002  P(feas)=0.4049
  p=3: ratio=0.9446  P(opt)=0.0002  P(feas)=0.3113
  p=4: ratio=0.9250  P(opt)=0.0002  P(feas)=0.4777
